# 2. RQ1: Adoption and agent authorship

**Methodology step 3: quantitative descriptive analysis.**

* RQ1.1 How fast did the format spread after October 2025?
* RQ1.2 What share of skills were first committed with an AI assistant credited as co-author (a `Co-authored-by` trailer) or by a bot account?

Proportions are reported with 95% Wilson confidence intervals.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("GITSKILLS_DB", str(ROOT / "data/raw/agent_skills_release.db"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from gitskills import clean, db, reuse, adoption, patterns
from gitskills.plots import save, save_table
print("database:", os.environ["GITSKILLS_DB"])
from statsmodels.stats.proportion import proportion_confint

In [ ]:
skills = clean.strict(clean.load_skills())
cov = adoption.history_coverage(skills)
complete = cov.loc[cov["share"] >= 0.9, "location_class"].tolist()
print("location classes with complete history:", complete)

## 2.1 New skills per month (complete-history classes only)

In [ ]:
std = skills[skills["location_class"].isin(complete)]
cohorts = adoption.monthly_cohorts(std)
ax = cohorts.set_index("first_month")["new_skills"].plot(marker=".", figsize=(6, 2.6))
ax.set_ylabel("new distinct skills"); plt.show()
cohorts

## 2.2 Assistant authorship

In [ ]:
h = skills[skills["has_history"]].copy()
h["ai_first"] = h["first_commit_ai"].notna() | h["first_commit_bot"]
k, n = int(h["ai_first"].sum()), len(h)
lo, hi = proportion_confint(k, n, method="wilson")
print(f"first commit by an assistant or bot: {k:,}/{n:,} = {k/n:.1%} (95% CI {lo:.1%}-{hi:.1%})")

In [ ]:
h["first_commit_ai"].value_counts().rename("skills").to_frame()

In [ ]:
by_month = h.groupby("first_month")["ai_first"].agg(skills="size", share="mean")
ax = by_month["share"].plot(marker=".", figsize=(6, 2.6)); ax.set_ylabel("share assistant-authored"); plt.show()
by_month